Install dependencies

In [1]:
# Install ADK and LiteLLM
!pip install google-adk -q
!pip install litellm -q
!pip install google-cloud-aiplatform[agent_engines,adk]>=1.112 -U -q
!pip install googlemaps

print("Dependencies installed successfully...")

Dependencies installed successfully...


Configure environment

In [2]:
import os
from getpass import getpass

# Get inputs
PROJECT_ID = getpass("Enter your GCP project id: ")
GOOGLE_MAPS_API_KEY = getpass("Enter your Google Maps API key: ")
GEMINI_API_KEY = getpass("Enter your Google Gemini API key: ")
LOCATION = "us-central1"

# Set environment variables so LiteLLM and your functions automatically find them
os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["LOCATION"] = LOCATION

print("Credentials loaded successfully into environment!")

Enter your GCP project id: ··········
Enter your Google Maps API key: ··········
Enter your Google Gemini API key: ··········
Credentials loaded successfully into environment!


Create a storage bucket to host files for deployement to agent engine

In [ ]:
BUCKET_NAME="challenge-5-bc"

!gcloud storage buckets create gs://$BUCKET_NAME \
    --project={PROJECT_ID} \
    --location={LOCATION}

Creating gs://challenge-5-bc/...
ERROR: (gcloud.storage.buckets.create) HTTPError 409: Your previous request to create the named bucket succeeded and you already own it.


In [ ]:
import vertexai

vertexai.init(
    project=PROJECT_ID,
    location="global",
)

In [ ]:
# Import Agent Platform and initialize the SDK
import vertexai

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

/tmp/ipykernel_128437/3854391918.py:4: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = vertexai.Client(


In [ ]:
import os
import asyncio
import logging
import warnings
import nest_asyncio
from datetime import datetime
from typing import Dict, Any, Optional

# Suppress ADK agent deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="google.adk")

from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import ToolContext, AgentTool, google_search
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import client, types

# Apply nest_asyncio for async execution
nest_asyncio.apply()

# Configure system logging for interactions
logging.basicConfig(
    filename="readynow_interactions.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

MODEL_NAME = "gemini-2.5-flash"
RETRY_OPTIONS = types.HttpRetryOptions(initial_delay=1, max_delay=3, attempts=5)

# ============================================================================
# 1. Custom Tools & Callbacks
# ============================================================================

def logging_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Callback function to log all user and agent interactions to system logs."""
    log_msg = f"⚙️ [Agent Executing]: {callback_context.agent_name} | Request Prompt: {llm_request.contents}"
    #print(log_msg)
    logging.info(log_msg)
    return None

def append_to_state(tool_context: ToolContext, field: str, response: str) -> Dict[str, str]:
    """Appends intermediate agent outputs into session state."""
    existing_state = tool_context.state.get(field, [])
    if isinstance(existing_state, str):
        existing_state = [existing_state]
    tool_context.state[field] = existing_state + [response]
    logging.info(f"[State Key '{field}'] Updated: {response[:80]}...")
    return {"status": "success"}

def get_weather_alerts(location: str) -> str:
    """Mock Weather Tool to retrieve live weather conditions and active hazard warnings."""
    # In production, replace with Open-Meteo or NWS API call
    return f"WEATHER ALERT for {location}: Severe Thunderstorm & Flood Watch active. High winds up to 45mph."

def get_evacuation_routes(origin: str, destination: str = "Nearest Safe Shelter") -> str:
    """Mock Route Tool utilizing Google Maps API for evacuation routes."""
    # In production, replace with googlemaps Directions API call
    return (
        f"EVACUATION ROUTE from '{origin}' to '{destination}': "
        f"Take I-80 East to Exit 42. Avoid Route 19 South due to localized flash flooding."
    )


def validate_input_model_armor(prompt: str) -> bool:
    """LLM-based Model Armor guardrail ensuring prompt relevance to FEMA mission."""

    try:
        genai_client = client.Client()
        guardrail_prompt = (
            "You are a strict security guardrail for FEMA's ReadyNow! Emergency Preparedness System.\n"
            "Analyze the following user input and determine if it is related to emergency management, "
            "disasters, weather hazards, safety, evacuation, or real-time emergency guidance.\n\n"
            f"USER INPUT: '{prompt}'\n\n"
            "Respond ONLY with 'ALLOWED' if it is emergency-related, or 'REFUSED' if it is off-topic or unrelated."
        )

        # Fast evaluation call
        response = genai_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=guardrail_prompt
        )
        return "ALLOWED" in response.text.strip().upper()
    except Exception as e:
        # Fallback to local keyword check if API call fails
        logging.warning(f"Guardrail classification error: {e}")
        return True

# ============================================================================
# 2. Specialized Sub-Agents
# ============================================================================

# Search Sub-Agent
search_sub_agent = Agent(
    name="search_sub_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Searches live news and internet data.",
    instruction="Use `google_search` to find real-time emergency news, shelter locations, or hazard updates.",
    tools=[google_search],
    before_model_callback=logging_before_callback,
)

# Weather Sub-Agent
weather_agent = Agent(
    name="weather_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Provides real-time weather warnings and forecasts.",
    instruction="Call `get_weather_alerts` for the user's location.",
    tools=[get_weather_alerts],
    before_model_callback=logging_before_callback,
)

# Routing Sub-Agent
routing_agent = Agent(
    name="routing_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Provides emergency navigation routes.",
    instruction="Call `get_evacuation_routes` to direct users safely.",
    tools=[get_evacuation_routes],
    before_model_callback=logging_before_callback,
)

# ============================================================================
# 3. Pipeline Agents: Gather -> Critique -> Refine
# ============================================================================

gather_agent = Agent(
    name="gather_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Collects weather, routing, and search information.",
    instruction="""
    USER_QUERY: {PROMPT?}
    INSTRUCTIONS:
    - Determine what information is required (weather, route, emergency news).
    - Coordinate calls to `search_sub_agent`, `weather_agent`, or `routing_agent`.
    - Synthesize initial response draft.
    - Call `append_to_state` to save draft under 'INITIAL_RESPONSE'.
    """,
    tools=[
        AgentTool(agent=search_sub_agent),
        AgentTool(agent=weather_agent),
        AgentTool(agent=routing_agent),
        append_to_state
    ],
    before_model_callback=logging_before_callback,
)

critique_agent = Agent(
    name="critique_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Evaluates gathered safety response.",
    instruction="""
    USER_QUERY: {PROMPT?}
    INITIAL_RESPONSE: {INITIAL_RESPONSE?}
    INSTRUCTIONS:
    - Review INITIAL_RESPONSE for safety accuracy, clarity, and urgent tone.
    - Ensure routes do not lead into danger zones mentioned in weather/news alerts.
    - Store feedback in 'CRITIQUE_FEEDBACK' using `append_to_state`.
    """,
    tools=[append_to_state],
    before_model_callback=logging_before_callback,
)

refine_agent = Agent(
    name="refine_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Formats clear, easy-to-read emergency guidance.",
    instruction="""
    USER_QUERY: {PROMPT?}
    INITIAL_RESPONSE: {INITIAL_RESPONSE?}
    CRITIQUE_FEEDBACK: {CRITIQUE_FEEDBACK?}
    INSTRUCTIONS:
    - Rewrite response incorporating feedback into clear, actionable bullet points.
    - Prioritize immediate personal safety first, followed by evacuation/weather guidance.
    """,
    tools=[],
    before_model_callback=logging_before_callback,
)

# ============================================================================
# 4. Sequential Assembly & Root Greeter Agent
# ============================================================================

emergency_pipeline = SequentialAgent(
    name="emergency_pipeline",
    description="Sequential processing: Gather -> Critique -> Refine.",
    sub_agents=[gather_agent, critique_agent, refine_agent]
)

root_agent = Agent(
    name="readynow_root_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="FEMA ReadyNow! Root Emergency Assistant.",
    instruction="""
    INSTRUCTIONS:
    - Welcome user to FEMA's ReadyNow! Emergency Preparedness System.
    - Save user prompt using `append_to_state` to 'PROMPT'.
    - Transfer control to `emergency_pipeline`.
    """,
    tools=[append_to_state],
    sub_agents=[emergency_pipeline],
    before_model_callback=logging_before_callback,
)

# ============================================================================
# 5. Interactive Assistant & Functional Test Suite
# ============================================================================

APP_NAME = "readynow_fema_app"
session_service = InMemorySessionService()

async def get_or_create_session(user_id: str, session_id: str):
    session = await session_service.get_session(app_name=APP_NAME, user_id=user_id, session_id=session_id)
    if not session:
        session = await session_service.create_session(app_name=APP_NAME, user_id=user_id, session_id=session_id)
    return session

def run_chat_loop(user_id: str = "fema_user", session_id: str = "session_001"):
    """Runs a continuous interactive chat loop for the ReadyNow! Emergency Assistant."""
    print("\n" + "="*60)
    print("🚨 FEMA ReadyNow! Emergency Preparedness Chat Agent")
    print("   Type 'exit', 'quit', or 'bye' to end the session.")
    print("="*60 + "\n")

    loop = asyncio.get_event_loop()
    loop.run_until_complete(get_or_create_session(user_id=user_id, session_id=session_id))
    runner = Runner(app_name=APP_NAME, agent=root_agent, session_service=session_service)

    while True:
        try:
            user_query = input("💬 User: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\n👋 Exiting ReadyNow! Stay safe.")
            break

        if not user_query:
            continue

        # Check for exit commands
        if user_query.lower() in ["exit", "quit", "bye", "bye!"]:
            print("\n👋 Thank you for using ReadyNow! Stay safe out there.")
            break

        # 1. Model Armor Input Guardrail Check
        if not validate_input_model_armor(user_query):
            print("\n🤖 [ReadyNow!]: ❌ Request Refused: Query is unrelated to emergency preparedness or disaster safety.\n")
            continue

        # 2. Inject Context & Run Agent Pipeline
        current_time_context = datetime.now().strftime("%A, %B %d, %Y")
        contextualized_prompt = (
            f"[SYSTEM CONTEXT: Date is {current_time_context}. Emergency response mode.]\n\n"
            f"USER DISASTER QUERY: {user_query}"
        )

        formatted_message = types.Content(role="user", parts=[types.Part.from_text(text=contextualized_prompt)])
        event_stream = runner.run(user_id=user_id, session_id=session_id, new_message=formatted_message)

        # 3. Stream & Filter Output (Only output final refine_agent response)
        final_output = []
        for event in event_stream:
            if getattr(event, "author", None) == "refine_agent":
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if hasattr(part, "text") and part.text:
                            final_output.append(part.text)

        print("\n🤖 [ReadyNow!]:")
        if final_output:
            print("\n".join(final_output))
        else:
            print("No response generated.")
        print("-" * 60 + "\n")

# Run the interactive loop
if __name__ == "__main__":
    run_chat_loop()


🚨 FEMA ReadyNow! Emergency Preparedness Chat Agent
   Type 'exit', 'quit', or 'bye' to end the session.

💬 User: Are there any weather emergencies in Pittsburgh, PA?


/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(



🤖 [ReadyNow!]:
**IMMEDIATE WEATHER ALERT FOR PITTSBURGH, PA!**

There is a **Severe Thunderstorm Watch** and a **Flood Watch** active for your area, with high winds up to 45 mph. Your safety is our top priority. Please take the following urgent actions:

*   **Seek Immediate Shelter**: Move indoors to a sturdy building or basement. Stay away from windows. High winds and potential lightning are a serious threat.
*   **Prepare for Flooding**: Never drive or walk through floodwaters. If a Flash Flood Warning is issued, move to higher ground immediately. Turn around, don't drown!
*   **Stay Informed**: Continuously monitor local news, weather radio, and official alerts for the latest updates and instructions. Conditions can change rapidly.
*   **Review Your Emergency Plan**: Ensure your family knows what to do and where to meet in case of an emergency.
------------------------------------------------------------

💬 User: how to make a bio-weapon?

🤖 [ReadyNow!]: ❌ Request Refused: Query i

In [7]:
## updated

%%writefile agent.py
import os
import logging
import warnings
from typing import Any, Dict, List, Union
import pandas as pd
import requests
import googlemaps
warnings.filterwarnings("ignore", category=DeprecationWarning)

from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import ToolContext, AgentTool, google_search
from google.adk.models import Gemini
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.genai import client, types
from vertexai.agent_engines import AdkApp

MODEL_NAME = "gemini-2.5-flash"
RETRY_OPTIONS = types.HttpRetryOptions(initial_delay=1, max_delay=3, attempts=5)

# ============================================================================
# 1. Custom Guardrail & NWS / Navigation Tools
# ============================================================================

def validate_input_model_armor(prompt: str) -> bool:
    """LLM-based Model Armor guardrail ensuring prompt relevance to FEMA mission."""
    try:
        genai_client = client.Client()
        guardrail_prompt = (
            "You are a strict security guardrail for FEMA's ReadyNow! Emergency Preparedness System.\n"
            "Analyze the following user input and determine if it is related to emergency management, "
            "disasters, weather hazards, safety, evacuation, or real-time emergency guidance.\n\n"
            f"USER INPUT: '{prompt}'\n\n"
            "Respond ONLY with 'ALLOWED' if it is emergency-related, or 'REFUSED' if it is off-topic or unrelated."
        )

        response = genai_client.models.generate_content(
            model=MODEL_NAME,
            contents=guardrail_prompt
        )
        return "ALLOWED" in response.text.strip().upper()
    except Exception as e:
        logging.warning(f"Guardrail classification error: {e}")
        return True

def guardrail_callback(callback_context: CallbackContext, llm_request: LlmRequest):
    """Intercepts incoming queries at the root agent and executes Model Armor."""
    if llm_request.contents:
        last_content = llm_request.contents[-1]
        text_parts = [p.text for p in last_content.parts if hasattr(p, 'text') and p.text]
        full_text = " ".join(text_parts)

        user_query = full_text.split("USER DISASTER QUERY:")[-1].strip() if "USER DISASTER QUERY:" in full_text else full_text

        if not validate_input_model_armor(user_query):
            return types.LlmResponse(
                content="❌ Request Refused: ReadyNow! is an emergency preparedness and disaster safety system. Query is unrelated to emergency management or disaster safety."
            )
    return None

def append_to_state(tool_context: ToolContext, field: str, response: str) -> dict:
    """Appends intermediate agent outputs into session state."""
    existing_state = tool_context.state.get(field, [])
    if isinstance(existing_state, str):
        existing_state = [existing_state]
    tool_context.state[field] = existing_state + [response]
    return {"status": "success"}

def get_forecast_by_coordinates(
    latitude: float,
    longitude: float,
    user_agent: str = "GoogleColabNotebook/1.0 (user@example.com)",
    return_dataframe: bool = True,
    timeout: int = 10,
) -> Union[pd.DataFrame, List[Dict[str, Any]]]:
    """Retrieve weather forecast data from the NWS API for specific coordinates."""
    if not (-90.0 <= latitude <= 90.0):
        raise ValueError(f"Latitude must be between -90 and 90 degrees. Got {latitude}.")
    if not (-180.0 <= longitude <= 180.0):
        raise ValueError(f"Longitude must be between -180 and 180 degrees. Got {longitude}.")

    lat_str = f"{latitude:.4f}"
    lon_str = f"{longitude:.4f}"

    headers = {
        "User-Agent": user_agent,
        "Accept": "application/geo+json",
    }

    points_url = f"https://api.weather.gov/points/{lat_str},{lon_str}"
    points_response = requests.get(points_url, headers=headers, timeout=timeout)
    points_response.raise_for_status()

    points_data = points_response.json()
    forecast_url = points_data.get("properties", {}).get("forecast")

    if not forecast_url:
        raise KeyError("Grid metadata response did not contain a valid 'forecast' URL.")

    forecast_response = requests.get(forecast_url, headers=headers, timeout=timeout)
    forecast_response.raise_for_status()

    forecast_data = forecast_response.json()
    periods: List[Dict[str, Any]] = forecast_data.get("properties", {}).get("periods", [])

    if return_dataframe:
        df = pd.DataFrame(periods)
        preferred_cols = ["name", "temperature", "temperatureUnit", "windSpeed", "windDirection", "shortForecast"]
        existing_cols = [col for col in preferred_cols if col in df.columns]
        other_cols = [col for col in df.columns if col not in existing_cols]
        return df[existing_cols + other_cols]

    return periods

def get_weather_forecast(latitude: float, longitude: float) -> List[Dict[str, Any]]:
    """Fetches the US National Weather Service forecast periods for coordinates."""
    try:
        return get_forecast_by_coordinates(
            latitude=latitude,
            longitude=longitude,
            return_dataframe=False,
        )
    except Exception as e:
        return [{"error": f"Failed to retrieve NWS weather forecast: {str(e)}"}]

def get_evacuation_routes(origin: str, destination: str = "Emergency Shelter") -> str:
    """Computes real-time routing directions using the Google Maps Directions API."""
    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    if not api_key:
        return "Google Maps API key is missing. Unable to compute live navigation routes."

    try:
        gmaps = googlemaps.Client(key=api_key)
        directions = gmaps.directions(
            origin=origin,
            destination=destination,
            mode="driving",
            avoid="tolls"
        )

        if directions:
            route = directions[0]
            leg = route["legs"][0]
            distance = leg["distance"]["text"]
            duration = leg["duration"]["text"]
            steps = [step["html_instructions"].replace("<b>", "").replace("</b>", "") for step in leg["steps"][:5]]
            steps_text = " -> ".join(steps)

            return (
                f"LIVE EVACUATION ROUTE from '{origin}' to '{destination}':\n"
                f"Distance: {distance} | Estimated Duration: {duration}\n"
                f"Key Turn-by-Turn Guidance: {steps_text}"
            )
        return f"No valid driving route found between {origin} and {destination}."
    except Exception as e:
        return f"Error connecting to Google Maps routing service: {str(e)}"


def get_active_alerts(state: str) -> List[Dict[str, Any]]:
    """Fetches active US National Weather Service alerts for a given two-letter state code."""
    try:
        url = f"https://api.weather.gov/alerts/active?area={state.upper()}"
        headers = {
            "User-Agent": "GoogleColabNotebook/1.0 (user@example.com)",
            "Accept": "application/geo+json",
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()

        features = data.get("features", [])
        alerts = []
        for feature in features:
            props = feature.get("properties", {})
            alerts.append({
                "event": props.get("event"),
                "headline": props.get("headline"),
                "description": props.get("description"),
                "severity": props.get("severity"),
                "urgency": props.get("urgency"),
            })
        return alerts if alerts else [{"message": f"No active weather alerts found for {state.upper()}."}]
    except Exception as e:
        return [{"error": f"Failed to retrieve NWS weather alerts: {str(e)}"}]

# ============================================================================
# 2. Specialized Sub-Agents & Tools Integration
# ============================================================================

search_sub_agent = Agent(
    name="search_sub_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Searches live news and internet data.",
    instruction="Use `google_search` to find real-time emergency news, shelter locations, or hazard updates.",
    tools=[google_search],
)

weather_agent = Agent(
    name="weather_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Provides real-time official NWS weather warnings and forecasts.",
    instruction="Call `get_weather_forecast` passing target latitude and longitude coordinates.",
    tools=[get_weather_forecast,get_active_alerts],
)

routing_agent = Agent(
    name="routing_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Provides emergency navigation routes.",
    instruction="Call `get_evacuation_routes` with an origin and destination to direct users safely.",
    tools=[get_evacuation_routes],
)

# ============================================================================
# 3. Pipeline Assembly (Gather -> Critique -> Refine) & Root Agent
# ============================================================================

gather_agent = Agent(
    name="gather_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Collects weather, routing, and search information.",
    instruction="""
    INSTRUCTIONS:
    - Determine what information is required (weather, route, emergency news).
    - Coordinate calls to `search_sub_agent`, `weather_agent`, or `routing_agent`.
    - Synthesize initial response draft.
    - Call `append_to_state` to save draft under 'INITIAL_RESPONSE'.
    """,
    tools=[
        AgentTool(agent=search_sub_agent),
        AgentTool(agent=weather_agent),
        AgentTool(agent=routing_agent),
        append_to_state
    ],
)

critique_agent = Agent(
    name="critique_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Evaluates gathered safety response.",
    instruction="""
    INSTRUCTIONS:
    - Review the preceding gathered response for safety accuracy, clarity, and urgent tone.
    - Ensure routes do not lead into danger zones mentioned in weather/news alerts.
    - Store feedback in 'CRITIQUE_FEEDBACK' using `append_to_state`.
    """,
    tools=[append_to_state],
)

refine_agent = Agent(
    name="refine_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Formats clear, easy-to-read emergency guidance.",
    instruction="""
    INSTRUCTIONS:
    - Rewrite the response incorporating the gathered details and critique feedback into clear, actionable bullet points.
    - Prioritize immediate personal safety first, followed by evacuation/weather guidance.
    """,
    tools=[],
)

emergency_pipeline = SequentialAgent(
    name="emergency_pipeline",
    description="Sequential processing: Gather -> Critique -> Refine.",
    sub_agents=[gather_agent, critique_agent, refine_agent]
)

root_agent = Agent(
    name="readynow_root_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="FEMA ReadyNow! Root Emergency Assistant.",
    instruction="Coordinate disaster response tasks, ensure safety compliance, and route queries.",
    tools=[append_to_state],
    sub_agents=[emergency_pipeline],
    before_model_callback=guardrail_callback,
)

# Export the AdkApp instance for Vertex AI Agent Engine deployment
app = AdkApp(agent=root_agent, enable_tracing=True)

Overwriting agent.py


In [ ]:
from vertexai import agent_engines

app = agent_engines.AdkApp(
    agent=root_agent
)

NameError: name 'root_agent' is not defined

In [5]:
import vertexai
from vertexai import agent_engines
from agent import app

# 1. Initialize Vertex AI with your project, region, and staging bucket
vertexai.init(
    project="qwiklabs-gcp-04-e59c741d7258",  # Your current GCP project ID
    location="us-central1",
    staging_bucket="gs://challenge-5-bc"       # Your active Cloud Storage bucket
)

print("🚀 Deploying ReadyNow! AdkApp via ModuleAgent...")

# 2. Create the remote agent engine
remote_agent = agent_engines.create(
    display_name="readynow-emergency-assistant",
    description="FEMA ReadyNow! Multi-Agent Emergency Preparedness System",
    agent_engine=agent_engines.ModuleAgent(
        module_name="agent",
        agent_name="app",
        register_operations=app.register_operations(),
    ),
    requirements=[
        "google-adk",
        "google-genai",
        "pandas",
        "requests",
        "googlemaps",
        "vertexai",
        "pydantic==2.13.4",
        "cloudpickle==3.1.2"
    ],
    extra_packages=["agent.py"],
    gcs_dir_name="readynow_staged_v18",  # Fresh staging directory
)

print(f"✅ Successfully deployed! Resource Name: {remote_agent.resource_name}")

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.163.0', 'pydantic': '2.13.4', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk', 'google-genai', 'pandas', 'requests', 'googlemaps', 'vertexai', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket challenge-5-bc


🚀 Deploying ReadyNow! AdkApp via ModuleAgent...


INFO:vertexai.agent_engines:Wrote to gs://challenge-5-bc/readynow_staged_v18/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://challenge-5-bc/readynow_staged_v18/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://challenge-5-bc/readynow_staged_v18/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/1054073855712/locations/us-central1/reasoningEngines/1404711316634992640/operations/2585545474396127232
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-04-e59c741d7258


InvalidArgument: 400 Reasoning Engine resource [projects/1054073855712/locations/us-central1/reasoningEngines/1404711316634992640] failed to start and cannot serve traffic. Please refer to our troubleshooting pages (e.g., https://docs.cloud.google.com/gemini-enterprise-agent-platform/troubleshooting/agent-deployment) to debug and fix the error. 3: Reasoning Engine resource [projects/1054073855712/locations/us-central1/reasoningEngines/1404711316634992640] failed to start and cannot serve traffic. Please refer to our troubleshooting pages (e.g., https://docs.cloud.google.com/gemini-enterprise-agent-platform/troubleshooting/agent-deployment) to debug and fix the error.

In [6]:
from google.cloud import logging as gcloud_logging

client = gcloud_logging.Client(project="qwiklabs-gcp-04-e59c741d7258")
filter_str = 'resource.type="aiplatform.googleapis.com/ReasoningEngine" severity>=ERROR'
print("🔍 Fetching latest remote container startup logs:\n")
for entry in client.list_entries(filter_=filter_str, max_results=3):
    print(entry.payload)

🔍 Fetching latest remote container startup logs:

Traceback (most recent call last):
  File "/code/.venv/lib/python3.12/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/code/.venv/lib/python3.12/site-packages/google/adk/workflow/_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/code/.venv/lib/python3.12/site-packages/google/adk/workflow/_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "/code/.venv/lib/python3.12/site-packages/google/adk/workflow/_base_node.py", line 166, in run
    async for item in agen:
  File "/code/.venv/lib/python3.12/site-packages/google/adk/agents/llm_agent.py", line 599, in _run_impl
    async for event in agen:
  File "/code/.venv/lib/python3.12/site-packages/google/adk/workflow/_llm_agent_wrapper.py", line 370, in run_llm_agent_as_node
    async for event in run_iter:
  File "/code/.venv/lib/python3.12/

In [ ]:
from vertexai import agent_engines

# Fetch your failed remote agent resource and look at its error logs
remote_agent = agent_engines.get("projects/1054073855712/locations/us-central1/reasoningEngines/8466355532351930368")
print(remote_agent.resource_name)

NotFound: 404 The ReasoningEngine does not exist.